In [1]:
import json
import os
import time
import pytz
import requests
from pathlib import Path
from datetime import datetime,timezone,timedelta
from scipy.optimize import curve_fit
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from dotenv import load_dotenv
from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

load_dotenv()

EXCHANGE_URL  = "http://ec2-52-49-69-152.eu-west-1.compute.amazonaws.com/"   
USERNAME = os.getenv("EXCHANGE_USERNAME")
PASSWORD = os.getenv("EXCHANGE_PASSWORD")

LONDON_TZ = pytz.timezone('Europe/London')

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """伦敦15分钟天气预报, 返回一个包含以下列的DataFrame: 时间, 温度, 湿度."""
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "humidity": m["relative_humidity_2m"],
    })

In [3]:
def get_wx_spot():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    wx_spot = weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["temperature"] * weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["humidity"]
    return round(wx_spot.item())

def get_wx_sum():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    weather_df["temp_hum_product"] = weather_df["temperature"]*weather_df["humidity"]
    wx_sum = weather_df["temp_hum_product"].sum()/100
    return round(wx_sum.item())

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

def get_thames(limit=400):
    """获取威斯敏斯特地区泰晤士河的最新潮汐数据. 返回一个包含以下列的DataFrame: 时间, 水平"""
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [ ]:
# 潮汐调和分析 - 5个主要分潮
# M2(12.42h), S2(12.00h), N2(12.66h), K1(23.93h), O1(25.82h)
TIDAL_CONSTITUENTS = [
    ('M2', 2*np.pi/12.4206),
    ('S2', 2*np.pi/12.0000),
    ('N2', 2*np.pi/12.6583),
    ('K1', 2*np.pi/23.9345),
    ('O1', 2*np.pi/25.8194),
]

def fit_tidal_harmonic(thames_df):
    """
    调和分析: h(t) = Z0 + sum_i [A_i*cos(w_i*t) + B_i*sin(w_i*t)]
    用最小二乘法拟合，返回 (Z0, [(A1,B1), (A2,B2), ...]) 和残差std.
    """
    t0 = thames_df['time'].min()
    hours = (thames_df['time'] - t0).dt.total_seconds().values / 3600
    y = thames_df['level'].values
    # 构建设计矩阵: [1, cos(w1*t), sin(w1*t), cos(w2*t), sin(w2*t), ...]
    cols = [np.ones(len(hours))]
    for _, w in TIDAL_CONSTITUENTS:
        cols.append(np.cos(w * hours))
        cols.append(np.sin(w * hours))
    X = np.column_stack(cols)
    # 最小二乘
    coeffs, residuals, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coeffs
    res_std = float(np.std(y - y_pred))
    return coeffs, t0, res_std

def predict_tidal_level(coeffs, t0, target_time):
    """用调和系数预测指定时刻的水位"""
    h = (target_time - t0).total_seconds() / 3600
    val = coeffs[0]
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        val += coeffs[1 + 2*i] * np.cos(w * h)
        val += coeffs[2 + 2*i] * np.sin(w * h)
    return float(val)

def predict_tidal_series(coeffs, t0, times):
    """批量预测一组时刻的水位"""
    hours = np.array([(t - t0).total_seconds() / 3600 for t in times])
    vals = np.full(len(hours), coeffs[0])
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        vals += coeffs[1 + 2*i] * np.cos(w * hours)
        vals += coeffs[2 + 2*i] * np.sin(w * hours)
    return vals

# 保留旧函数兼容性
def sin_function(x, a, b, c, d):
    return a + b * np.sin(c * x + d)

def fit_thames_data(thames_df):
    start_time = thames_df['time'].min()
    hours = (thames_df['time'] - start_time).dt.total_seconds() / 3600
    x_data, y_data = hours.values, thames_df['level'].values
    p0 = [np.mean(y_data), np.std(y_data), 2*np.pi/12.4206, 0]
    popt, _ = curve_fit(sin_function, x_data, y_data, p0=p0, maxfev=5000)
    return tuple(popt)


In [6]:
def get_tide_spot():
    thames_df = get_thames()
    a,b,c,d = fit_thames_data(thames_df)
    predicted_level = sin_function((LONDON_TZ.localize(datetime(2026,3,1,12,0,0))-thames_df["time"].min()).total_seconds()/3600,a,b,c,d)
    tide_spot = abs(predicted_level)*1000
    return round(tide_spot.item())

def get_tide_swing():
    thames_df = get_thames()
    thames_df = thames_df[thames_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0))]
    a,b,c,d = fit_thames_data(thames_df)
    
    date_range = pd.date_range(start=max(thames_df['time'])+timedelta(minutes=15),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="15min",tz="Europe/London")
    thames_missing_df = pd.DataFrame({'time':date_range})
    hours_since_start = (thames_missing_df["time"]-thames_df["time"].min()).dt.total_seconds()/3600
    thames_missing_df["level"] = a+b*np.sin(c*hours_since_start.values+d)
    thames_df = pd.concat([thames_df,thames_missing_df], ignore_index=True).sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)

    thames_df["level_cm"] = round(thames_df["level"]*100)
    thames_df["diff_cm"] = thames_df["level_cm"].diff().abs().dropna().reset_index(drop=True)
    thames_df["put_payoff"] = np.maximum(0,20-thames_df["diff_cm"])
    thames_df["call_payoff"] = np.maximum(0,thames_df["diff_cm"]-25)
    thames_df["total_payoff"] = thames_df["put_payoff"]+thames_df["call_payoff"]

    return thames_df["total_payoff"].sum()


In [7]:
AERODATABOX_KEY = os.getenv("AERODATABOX_KEY")
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_flights(airport=AIRPORT,offset_minutes=-360,duration_minutes=720,filters:dict|None=None):
    """按相对时间窗口(从现在开始的偏移量)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = f"?offsetMinutes={offset_minutes}&durationMinutes={duration_minutes}&direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

def fetch_flights_range(airport=AIRPORT,from_local="2026-02-28T12:00",to_local="2026-02-29T00:00",filters: dict|None = None):
    """根据明确的本地时间范围(最大跨度为12小时)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = "?direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}/{from_local}/{to_local}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

In [8]:
flights_data_1 = fetch_flights_range(from_local="2026-02-28T12:00",to_local="2026-03-01T00:00")

In [9]:
flights_data_2 = fetch_flights_range(from_local="2026-03-01T00:00",to_local="2026-03-01T12:00")

In [10]:
def get_lhr_count():
    global flights_data_1, flights_data_2    
    lhr_count = len(flights_data_1.get('arrivals',[]))+len(flights_data_1.get('departures',[]))+len(flights_data_2.get('arrivals',[]))+len(flights_data_2.get('departures',[]))
    return lhr_count

In [ ]:
def parse_flight_time(flight_record, flight_type):
    """解析航班时间. 优先用实际跑道时间, 其次修订时间, 最后计划时间."""
    time_fields = ['runwayTime', 'revisedTime', 'scheduledTime']
    movement = flight_record['movement']
    utc_str = None
    for field in time_fields:
        if field in movement and movement[field].get('utc'):
            utc_str = movement[field]['utc']
            break
    if utc_str is None:
        raise ValueError('No valid time field found')
    dt = pytz.utc.localize(datetime.strptime(utc_str, '%Y-%m-%d %H:%MZ'))
    return dt.astimezone(LONDON_TZ)

# 保留旧名兼容
flight_time = parse_flight_time

def group_flights(flights_data, start_time, end_time):
    """按时间窗口统计到达/出发航班数"""
    arrivals = 0
    for arr in flights_data.get('arrivals', []):
        try:
            if start_time <= parse_flight_time(arr, 'arrival') < end_time:
                arrivals += 1
        except Exception:
            pass
    departures = 0
    for dep in flights_data.get('departures', []):
        try:
            if start_time <= parse_flight_time(dep, 'departure') < end_time:
                departures += 1
        except Exception:
            pass
    return arrivals, departures


In [12]:
def get_settlement_price(symbol):
    price_map = {
        "TIDE_SPOT": get_tide_spot,
        "TIDE_SWING": get_tide_swing,
        "WX_SPOT": get_wx_spot,
        "WX_SUM": get_wx_sum,
        "LHR_COUNT": get_lhr_count,
        "LHR_INDEX": get_lhr_index,
        "LON_ETF": get_lon_etf,
        "LON_FLY": get_lon_fly,
    }
    fn = price_map.get(symbol)
    return fn() if fn else 10.0

def get_lon_etf():
    return get_tide_spot()+get_wx_spot()+get_lhr_count()

def get_lon_fly():
    etf_price = get_lon_etf() 
    part1 = 2*max(6200-etf_price,0)
    part2 = max(etf_price-6200,0)
    part3 = -2*max(etf_price-6600,0)
    part4 = 3*max(etf_price-7000,0)
    return part1 + part2 + part3 + part4

In [ ]:
class FairValueEngine:
    """
    Fair value 引擎:
    - 潮汐: 5分潮调和分析 (M2/S2/N2/K1/O1)
    - LON_FLY: 蒙特卡洛定价
    - 贝叶斯更新: 融合模型预测和市场价格
    - 数据缓存: 避免重复 API 调用
    """
    CACHE_TTL = 60
    SETTLE_TIME = datetime(2026, 3, 1, 12, 0, 0)
    SESSION_START = datetime(2026, 2, 28, 12, 0, 0)
    MC_SAMPLES = 20000

    def __init__(self):
        self._weather_cache = None
        self._weather_ts = 0.0
        self._thames_cache = None
        self._thames_ts = 0.0
        self._tidal_coeffs = None   # (coeffs, t0, res_std)
        self._flights_cache = None
        # 贝叶斯先验权重 (0=纯模型, 1=纯市场)
        self.market_weight = 0.2

    # ── 数据层 ────────────────────────────────────────────────────────────────
    def _get_weather(self):
        if time.time() - self._weather_ts > self.CACHE_TTL:
            self._weather_cache = get_weather(past_steps=96, forecast_steps=96)
            self._weather_ts = time.time()
        return self._weather_cache

    def _get_thames(self):
        if time.time() - self._thames_ts > self.CACHE_TTL:
            df = get_thames(limit=672)  # ~7天历史
            self._thames_cache = df
            self._tidal_coeffs = fit_tidal_harmonic(df)
            self._thames_ts = time.time()
        return self._thames_cache, self._tidal_coeffs

    def init_flights(self):
        if self._flights_cache is None:
            print('Fetching flight data (2 API calls)...')
            d1 = fetch_flights_range(from_local='2026-02-28T12:00', to_local='2026-03-01T00:00')
            d2 = fetch_flights_range(from_local='2026-03-01T00:00', to_local='2026-03-01T12:00')
            self._flights_cache = (d1, d2)
            print(f'  Loaded {get_lhr_count_from(d1, d2)} flights')
        return self._flights_cache

    # ── 贝叶斯融合 ────────────────────────────────────────────────────────────
    def bayesian_update(self, model_fv, model_std, market_mid):
        """
        融合模型预测和市场价格.
        posterior = weighted average, weight by precision (1/var).
        """
        if market_mid is None:
            return model_fv, model_std
        # 假设市场价格的不确定性 = model_std * 2 (市场可能有噪音)
        market_std = model_std * 2.0
        w_model = 1.0 / (model_std ** 2)
        w_market = 1.0 / (market_std ** 2)
        posterior_fv = (w_model * model_fv + w_market * market_mid) / (w_model + w_market)
        posterior_std = (1.0 / (w_model + w_market)) ** 0.5
        return posterior_fv, posterior_std

    # ── 各产品 fair value ─────────────────────────────────────────────────────
    def wx_spot_fv(self):
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod'] = df['temp_f'] * df['humidity']
        row = df[df['time'] == settle]
        if row.empty:
            row = df.iloc[[-1]]
        fv = float(row['prod'].values[0])
        std = float(df['prod'].std()) if len(df) > 1 else fv * 0.02
        return round(fv), max(std, 1.0)

    def wx_sum_fv(self):
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod'] = df['temp_f'] * df['humidity']
        now_utc = datetime.now(timezone.utc)
        now_lon = now_utc.astimezone(pytz.timezone('Europe/London'))
        observed = float(df[df['time'] <= now_lon]['prod'].sum() / 100)
        forecast_df = df[df['time'] > now_lon]
        forecast = float(forecast_df['prod'].sum() / 100)
        fv = observed + forecast
        per_step_std = float(df['prod'].std()) / 100 if len(df) > 1 else 5.0
        std = per_step_std * max(len(forecast_df) ** 0.5, 1)
        return round(fv), max(std, 1.0)

    def tide_spot_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        fv = abs(predict_tidal_level(coeffs, t0, settle)) * 1000
        std = res_std * 1000  # 残差 std 传播到 mm
        return round(fv), max(std, 1.0)

    def tide_swing_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        start = LONDON_TZ.localize(self.SESSION_START)
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        # 生成完整24h时间序列
        dr = pd.date_range(start=start, end=settle, freq='15min', tz='Europe/London')
        levels = predict_tidal_series(coeffs, t0, dr)
        # 用实测数据覆盖已有部分
        df_sess = df[df['time'] >= start].copy()
        level_series = pd.Series(levels, index=dr)
        for _, row in df_sess.iterrows():
            if row['time'] in level_series.index:
                level_series[row['time']] = row['level']
        diff_cm = level_series.diff().abs() * 100
        payoff = (np.maximum(0, 20 - diff_cm) + np.maximum(0, diff_cm - 25)).sum()
        fv = float(payoff)
        # 蒙特卡洛估计不确定性: 对每个时间步加噪声
        n_steps = len(dr)
        noise = np.random.normal(0, res_std, (1000, n_steps))
        noisy_levels = levels[np.newaxis, :] + noise
        noisy_diff = np.abs(np.diff(noisy_levels, axis=1)) * 100
        noisy_payoff = (np.maximum(0, 20 - noisy_diff) + np.maximum(0, noisy_diff - 25)).sum(axis=1)
        std = float(noisy_payoff.std())
        return round(fv), max(std, 1.0)

    def lhr_count_fv(self):
        if self._flights_cache is None:
            return 1400, 50.0
        d1, d2 = self._flights_cache
        fv = float(get_lhr_count_from(d1, d2))
        return round(fv), 30.0

    def lhr_index_fv(self):
        if self._flights_cache is None:
            return 50, 20.0
        d1, d2 = self._flights_cache
        fv = float(get_lhr_index_from(d1, d2))
        return round(fv), 15.0

    def lon_etf_fv(self, ts=None, wx=None, lc=None):
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        fv = ts[0] + wx[0] + lc[0]
        std = (ts[1]**2 + wx[1]**2 + lc[1]**2) ** 0.5
        return round(fv), max(std, 1.0)

    def lon_fly_fv(self, etf_fv=None, etf_std=None):
        """蒙特卡洛定价 LON_FLY，正确处理折点处的非线性"""
        if etf_fv is None or etf_std is None:
            etf_fv, etf_std = self.lon_etf_fv()
        def fly_payoff_vec(e):
            return (2*np.maximum(0, 6200-e)
                    + np.maximum(0, e-6200)
                    - 2*np.maximum(0, e-6600)
                    + 3*np.maximum(0, e-7000))
        samples = np.random.normal(etf_fv, etf_std, self.MC_SAMPLES)
        samples = np.maximum(0, samples)  # 结算价非负
        payoffs = fly_payoff_vec(samples)
        fv = float(payoffs.mean())
        std = float(payoffs.std())
        return round(fv), max(std, 1.0)

    def get_all(self, market_mids=None):
        """
        计算所有产品 fair value.
        market_mids: {symbol: mid_price} 用于贝叶斯更新.
        避免重复计算基础产品.
        """
        if market_mids is None:
            market_mids = {}
        # 基础产品（无依赖）
        ts  = self.tide_spot_fv()
        tsw = self.tide_swing_fv()
        wx  = self.wx_spot_fv()
        wxs = self.wx_sum_fv()
        lc  = self.lhr_count_fv()
        li  = self.lhr_index_fv()
        # 派生产品（复用基础产品结果）
        etf = self.lon_etf_fv(ts=ts, wx=wx, lc=lc)
        fly = self.lon_fly_fv(etf_fv=etf[0], etf_std=etf[1])
        raw = {
            'TIDE_SPOT': ts, 'TIDE_SWING': tsw,
            'WX_SPOT': wx,   'WX_SUM': wxs,
            'LHR_COUNT': lc, 'LHR_INDEX': li,
            'LON_ETF': etf,  'LON_FLY': fly,
        }
        # 贝叶斯更新
        result = {}
        for sym, (fv, std) in raw.items():
            mid = market_mids.get(sym)
            post_fv, post_std = self.bayesian_update(fv, std, mid)
            result[sym] = (post_fv, post_std)
        return result


In [ ]:
def get_lhr_count_from(d1, d2):
    return (len(d1.get('arrivals', [])) + len(d1.get('departures', []))
            + len(d2.get('arrivals', [])) + len(d2.get('departures', [])))

def get_lhr_index_from(d1, d2):
    date_range = pd.date_range(
        start=LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0)),
        end=LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0)),
        freq='30min', tz='Europe/London')
    idx = 0.0
    for i in range(24):
        arr, dep = group_flights(d1, date_range[i], date_range[i+1])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    for i in range(24):
        arr, dep = group_flights(d2, date_range[i+24], date_range[i+25])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    return round(abs(idx))


In [ ]:
class AlgoBot(BaseBot):
    """
    四模块策略:
      A. Fair-value 做市 (spread = std * multiplier, 仓位偏斜)
      B. ETF 成分套利 (实时 SSE 触发 + 主循环双重检测)
      C. 方向性押注 (z-score > 2 触发 IOC)
      D. LON_FLY delta 对冲
    改进:
      - on_orderbook 实时检测套利和方向性机会
      - 贝叶斯 fair value 融合市场价格
      - 套利阈值覆盖实际交易成本
      - PnL 追踪和止损
      - 结算前30分钟主动平仓
    """
    SYMBOLS = ['TIDE_SPOT','TIDE_SWING','WX_SPOT','WX_SUM',
               'LHR_COUNT','LHR_INDEX','LON_ETF','LON_FLY']
    ETF_LEGS = ['TIDE_SPOT', 'WX_SPOT', 'LHR_COUNT']
    POSITION_LIMIT = 100
    SETTLE_TIME = datetime(2026, 3, 1, 12, 0, 0)

    # 策略参数
    MM_SPREAD_MULT = 1.5
    MM_MIN_SPREAD  = 3
    MM_BASE_QTY    = 10
    REQUOTE_THRESH = 2       # fair value 变化超过此值才重挂
    ARB_THRESH     = 25      # 套利阈值: 覆盖4条腿spread成本
    ARB_QTY        = 20
    DIR_Z_THRESH   = 2.0
    DIR_QTY        = 15
    STOP_LOSS      = -5000   # 止损线
    UNWIND_MINS    = 30      # 结算前N分钟开始平仓
    LOOP_INTERVAL  = 10

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fve = FairValueEngine()
        self.fair_values = {}      # {symbol: (fv, std)}
        self.market_mids = {}      # {symbol: mid} 实时更新
        self.last_quotes = {}      # {symbol: (bid, ask)}
        self.active_orders = {}    # {symbol: [order_id]}
        self._arb_cooldown = 0.0   # 套利冷却时间戳
        self._dir_cooldown = {}    # {symbol: timestamp}

    # ── SSE 回调 ──────────────────────────────────────────────────────────────
    def on_trades(self, trade):
        side = 'BOUGHT' if trade.buyer == self.username else 'SOLD'
        print(f'  FILL {side} {trade.volume}x {trade.product} @ {trade.price}')

    def on_orderbook(self, ob):
        """实时响应 orderbook 变化: 更新 mid, 检测套利和方向性机会"""
        mid = self._calc_mid(ob)
        if mid is not None:
            self.market_mids[ob.product] = mid
        # 实时套利检测 (ETF 相关产品变化时触发)
        if ob.product in self.ETF_LEGS + ['LON_ETF']:
            self._realtime_arb_check()
        # 实时方向性检测
        if ob.product in self.fair_values:
            self._realtime_dir_check(ob.product)

    # ── 工具方法 ──────────────────────────────────────────────────────────────
    def _calc_mid(self, ob):
        bids = [o.price for o in ob.buy_orders if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        return (max(bids) + min(asks)) / 2 if bids and asks else None

    def _send_ioc(self, order):
        resp = self.send_order(order)
        if resp and resp.status == 'ACTIVE':
            self.cancel_order(resp.id)
        return resp

    def _cancel_symbol(self, symbol):
        for oid in self.active_orders.get(symbol, []):
            self.cancel_order(oid)
        self.active_orders[symbol] = []

    def _mins_to_settle(self):
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        now = datetime.now(pytz.timezone('Europe/London'))
        return (settle - now).total_seconds() / 60

    def _get_positions(self):
        return self.get_positions()

    # ── 实时套利检测 (SSE 触发) ───────────────────────────────────────────────
    def _realtime_arb_check(self):
        if time.time() - self._arb_cooldown < 2.0:  # 2秒冷却
            return
        etf_mid = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        if abs(basis) > self.ARB_THRESH:
            positions = self._get_positions()
            self._execute_arb(basis, positions)
            self._arb_cooldown = time.time()

    # ── 实时方向性检测 (SSE 触发) ─────────────────────────────────────────────
    def _realtime_dir_check(self, symbol):
        if time.time() - self._dir_cooldown.get(symbol, 0) < 5.0:  # 5秒冷却
            return
        if symbol not in self.fair_values:
            return
        fv, std = self.fair_values[symbol]
        mid = self.market_mids.get(symbol)
        if mid is None or std <= 0:
            return
        z = (mid - fv) / std
        if abs(z) > self.DIR_Z_THRESH:
            positions = self._get_positions()
            self._execute_directional(symbol, z, fv, mid, std, positions)
            self._dir_cooldown[symbol] = time.time()

    # ── 模块 A: Fair-value 做市 ───────────────────────────────────────────────
    def module_a_market_make(self, positions):
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values:
                continue
            fv, std = self.fair_values[symbol]
            half_spread = max(self.MM_MIN_SPREAD, std * self.MM_SPREAD_MULT / 2)
            pos = positions.get(symbol, 0)
            # 仓位偏斜: 仓位越重, 报价越偏向平仓方向
            skew = (pos / self.POSITION_LIMIT) * half_spread * 0.5
            bid_price = round(fv - half_spread - skew)
            ask_price = round(fv + half_spread - skew)
            if bid_price <= 0 or ask_price <= bid_price:
                continue
            # 只在 fair value 变化超过阈值时重挂
            last = self.last_quotes.get(symbol)
            if (last and abs(last[0] - bid_price) < self.REQUOTE_THRESH
                    and abs(last[1] - ask_price) < self.REQUOTE_THRESH):
                continue
            self._cancel_symbol(symbol)
            avail_buy  = max(0, self.POSITION_LIMIT - pos)
            avail_sell = max(0, self.POSITION_LIMIT + pos)
            new_ids = []
            if avail_buy >= 1:
                qty = min(self.MM_BASE_QTY, avail_buy)
                r = self.send_order(OrderRequest(symbol, bid_price, Side.BUY, qty))
                if r: new_ids.append(r.id)
            if avail_sell >= 1:
                qty = min(self.MM_BASE_QTY, avail_sell)
                r = self.send_order(OrderRequest(symbol, ask_price, Side.SELL, qty))
                if r: new_ids.append(r.id)
            self.active_orders[symbol] = new_ids
            self.last_quotes[symbol] = (bid_price, ask_price)

    # ── 模块 B: ETF 套利 ──────────────────────────────────────────────────────
    def _execute_arb(self, basis, positions):
        """执行套利: 同时发出4条腿 IOC"""
        if basis > self.ARB_THRESH:
            # 卖 ETF, 买三个成分
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT + positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT - positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: return
            print(f'  ARB SELL ETF basis={basis:.1f} qty={qty}')
            ob_etf = self.get_orderbook('LON_ETF')
            if ob_etf.buy_orders:
                self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob_etf.buy_orders), Side.SELL, qty))
            for s in self.ETF_LEGS:
                ob = self.get_orderbook(s)
                if ob.sell_orders:
                    self._send_ioc(OrderRequest(s, min(o.price for o in ob.sell_orders), Side.BUY, qty))
        else:
            # 买 ETF, 卖三个成分
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT - positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT + positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: return
            print(f'  ARB BUY ETF basis={basis:.1f} qty={qty}')
            ob_etf = self.get_orderbook('LON_ETF')
            if ob_etf.sell_orders:
                self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob_etf.sell_orders), Side.BUY, qty))
            for s in self.ETF_LEGS:
                ob = self.get_orderbook(s)
                if ob.buy_orders:
                    self._send_ioc(OrderRequest(s, max(o.price for o in ob.buy_orders), Side.SELL, qty))

    def module_b_arbitrage(self, positions):
        etf_mid = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        if abs(basis) > self.ARB_THRESH:
            self._execute_arb(basis, positions)

    # ── 模块 C: 方向性押注 ────────────────────────────────────────────────────
    def _execute_directional(self, symbol, z, fv, mid, std, positions):
        pos = positions.get(symbol, 0)
        if z > self.DIR_Z_THRESH:
            qty = min(self.DIR_QTY, self.POSITION_LIMIT + pos)
            if qty < 1: return
            ob = self.get_orderbook(symbol)
            if ob.buy_orders:
                best_bid = max(o.price for o in ob.buy_orders)
                print(f'  DIR SELL {symbol} z={z:.2f} fv={fv:.0f} mid={mid:.0f}')
                self._send_ioc(OrderRequest(symbol, best_bid, Side.SELL, qty))
        elif z < -self.DIR_Z_THRESH:
            qty = min(self.DIR_QTY, self.POSITION_LIMIT - pos)
            if qty < 1: return
            ob = self.get_orderbook(symbol)
            if ob.sell_orders:
                best_ask = min(o.price for o in ob.sell_orders)
                print(f'  DIR BUY  {symbol} z={z:.2f} fv={fv:.0f} mid={mid:.0f}')
                self._send_ioc(OrderRequest(symbol, best_ask, Side.BUY, qty))

    def module_c_directional(self, positions):
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values: continue
            fv, std = self.fair_values[symbol]
            mid = self.market_mids.get(symbol)
            if mid is None or std <= 0: continue
            z = (mid - fv) / std
            if abs(z) > self.DIR_Z_THRESH:
                self._execute_directional(symbol, z, fv, mid, std, positions)

    # ── 模块 D: LON_FLY delta 对冲 ────────────────────────────────────────────
    def module_d_fly_hedge(self, positions):
        fly_pos = positions.get('LON_FLY', 0)
        if fly_pos == 0: return
        etf_mid = self.market_mids.get('LON_ETF')
        if etf_mid is None: return
        if   etf_mid < 6200: fly_delta = -2
        elif etf_mid < 6600: fly_delta = -1
        elif etf_mid < 7000: fly_delta =  1
        else:                fly_delta = -2
        target_hedge = -fly_pos * fly_delta
        current_etf  = positions.get('LON_ETF', 0)
        delta_needed = target_hedge - current_etf
        if abs(delta_needed) < 5: return
        qty = min(abs(delta_needed), self.POSITION_LIMIT - abs(current_etf))
        if qty < 1: return
        ob = self.get_orderbook('LON_ETF')
        if delta_needed > 0 and ob.sell_orders:
            print(f'  HEDGE BUY LON_ETF {qty}')
            self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob.sell_orders), Side.BUY, qty))
        elif delta_needed < 0 and ob.buy_orders:
            print(f'  HEDGE SELL LON_ETF {qty}')
            self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob.buy_orders), Side.SELL, qty))

    # ── 结算前平仓 ────────────────────────────────────────────────────────────
    def unwind_all(self, positions):
        print('  UNWIND: closing all positions before settlement')
        self.cancel_all_orders()
        for symbol, pos in positions.items():
            if pos == 0: continue
            ob = self.get_orderbook(symbol)
            if pos > 0 and ob.buy_orders:
                best_bid = max(o.price for o in ob.buy_orders)
                self._send_ioc(OrderRequest(symbol, best_bid, Side.SELL, abs(pos)))
            elif pos < 0 and ob.sell_orders:
                best_ask = min(o.price for o in ob.sell_orders)
                self._send_ioc(OrderRequest(symbol, best_ask, Side.BUY, abs(pos)))

    # ── 主循环 ────────────────────────────────────────────────────────────────
    def run_bot(self):
        print('Initialising...')
        self.fve.init_flights()
        self.start()  # 启动 SSE
        print('Bot running. Ctrl+C to stop.')
        try:
            while True:
                t0 = time.time()
                mins_left = self._mins_to_settle()

                # 结算前平仓
                if mins_left <= self.UNWIND_MINS:
                    positions = self._get_positions()
                    self.unwind_all(positions)
                    print(f'  {mins_left:.1f} mins to settle, unwinding. Sleeping 60s.')
                    time.sleep(60)
                    continue

                # 1. 更新市场 mid (补充 SSE 可能遗漏的)
                for sym in self.SYMBOLS:
                    try:
                        ob = self.get_orderbook(sym)
                        mid = self._calc_mid(ob)
                        if mid: self.market_mids[sym] = mid
                    except Exception:
                        pass

                # 2. 更新 fair values (传入市场 mid 做贝叶斯更新)
                self.fair_values = self.fve.get_all(market_mids=self.market_mids)
                fv_str = {s: f'{v[0]}±{v[1]:.0f}' for s, v in self.fair_values.items()}
                print(f'\n[{datetime.now().strftime("%H:%M:%S")}] {mins_left:.0f}min left')
                print(f'  FV: {fv_str}')

                # 3. 获取仓位 + PnL
                positions = self._get_positions()
                pnl = self.get_pnl()
                print(f'  Pos: {positions}  PnL: {pnl}')

                # 止损
                if pnl.get('unrealizedPnl', 0) < self.STOP_LOSS:
                    print(f'  STOP LOSS triggered at {pnl}')
                    self.cancel_all_orders()
                    self.stop()
                    break

                # 4. 套利 (主循环补充检测)
                self.module_b_arbitrage(positions)
                # 5. 方向性
                self.module_c_directional(positions)
                # 6. FLY 对冲
                self.module_d_fly_hedge(positions)
                # 7. 做市
                self.module_a_market_make(positions)

                elapsed = time.time() - t0
                time.sleep(max(1, self.LOOP_INTERVAL - elapsed))

        except KeyboardInterrupt:
            print('\nStopping...')
            self.cancel_all_orders()
            self.stop()
            print('Final positions:', self.get_positions())
            print('Final PnL:', self.get_pnl())
        except Exception as e:
            import traceback
            traceback.print_exc()
            self.cancel_all_orders()
            self.stop()


In [ ]:
bot = AlgoBot(EXCHANGE_URL, USERNAME, PASSWORD)
bot.run_bot()
